In [1]:
import pandas as pd

RAW_DIR = '../data/raw/'
PROCESSED_DIR = '../data/processed/'

In [2]:
df_deputados = pd.read_csv(f'{RAW_DIR}deputados.csv', sep=';')

df_votacoes = pd.read_csv(f'{RAW_DIR}votacoes-2025.csv', sep=';')

df_votos = pd.read_csv(f'{RAW_DIR}votacoesVotos-2025.csv', sep=';')

df_orientacao = pd.read_csv(f'{RAW_DIR}votacoesOrientacoes-2025.csv', sep=';')

In [3]:
df_deputados['deputado_id'] = df_deputados['uri'].str.rsplit('/', n=1).str[-1].astype(int)

deputados_features = [
    'deputado_id',
    'nome',
    'siglaSexo',
    'ufNascimento',
    'dataNascimento'
]

df_deputados_processed = df_deputados[df_deputados['idLegislaturaFinal'] == 57]

df_deputados_processed = df_deputados_processed[deputados_features]

latest_info = (
    df_votos
    .sort_values('dataHoraVoto')
    .drop_duplicates('deputado_id', keep='last')
    [['deputado_id', 'deputado_siglaPartido', 'deputado_siglaUf']]
)

df_deputados_processed = df_deputados_processed.merge(latest_info, how='left')

df_deputados_processed = df_deputados_processed.dropna(subset=['deputado_siglaPartido', 'deputado_siglaUf'])

df_deputados_processed['dataNascimento'] = pd.to_datetime(df_deputados_processed['dataNascimento'])

df_deputados_processed.head()

,deputado_id,nome,siglaSexo,ufNascimento,dataNascimento,deputado_siglaPartido,deputado_siglaUf
1,204379,Acácio Favacho,M,AP,1983-09-28,MDB,AP
2,220714,Adail Filho,M,AM,1992-02-16,REPUBLICANOS,AM
3,221328,Adilson Barroso,M,MG,1964-06-14,PL,SP
4,204560,Adolfo Viana,M,BA,1981-02-02,PSDB,BA
5,204528,Adriana Ventura,F,SP,1969-03-06,NOVO,SP


In [4]:
df_votacoes_processed = df_votacoes[df_votacoes['siglaOrgao'] == 'PLEN']

votacoes_features = [
    'id',
    'data',
    'dataHoraRegistro',
    'aprovacao',
    'votosSim',
    'votosNao',
    'votosOutros',
    'descricao',
    'ultimaApresentacaoProposicao_idProposicao'
]

df_votacoes_processed = df_votacoes_processed[votacoes_features]

df_votacoes_processed['data'] = pd.to_datetime(df_votacoes_processed['data'])

df_votacoes_processed['dataHoraRegistro'] = pd.to_datetime(df_votacoes_processed['dataHoraRegistro'])

df_votacoes_processed = df_votacoes_processed.rename(columns={'id': 'votacao_id'})

df_votacoes_processed = df_votacoes_processed[
    (df_votacoes_processed['votosSim'] + df_votacoes_processed['votosNao']) > 0
]

plen_vote_ids = set(df_votacoes_processed['votacao_id'])
print(f"Plenário vote IDs with actual votes: {len(plen_vote_ids)}")

df_votacoes_processed.head()

Plenário vote IDs with actual votes: 429


,votacao_id,data,dataHoraRegistro,aprovacao,votosSim,votosNao,votosOutros,descricao,ultimaApresentacaoProposicao_idProposicao
22,2162802-89,2025-02-11,2025-02-11 17:43:14,1.0,297,107,2,Aprovado o Substitutivo ao Projeto de Lei nº 9...,2483606
29,2438467-41,2025-02-12,2025-02-12 16:42:30,0.0,125,236,4,Rejeitado o Requerimento. Sim: 125; Não: 236; ...,0
30,2438467-47,2025-02-12,2025-02-12 19:54:47,1.0,273,136,1,"Aprovado o Projeto de Lei nº 2.215, de 2024. S...",2483860
52,2484059-7,2025-02-18,2025-02-18 18:32:01,1.0,266,140,2,Aprovado o Requerimento de Urgência (Art. 155 ...,0
55,2397890-54,2025-02-18,2025-02-18 19:49:56,0.0,34,356,2,Rejeitado o Substitutivo ao Projeto de Lei Com...,2484388


In [5]:
df_votos_processed = df_votos[df_votos['idVotacao'].isin(plen_vote_ids)]

votos_features = [
    'idVotacao',
    'deputado_id',
    'voto',
    'deputado_siglaPartido',
    'deputado_siglaUf',
    'dataHoraVoto'
]

df_votos_processed = df_votos_processed[votos_features]

df_votos_processed = df_votos_processed.drop_duplicates()

df_votos_processed = df_votos_processed.rename(columns={'idVotacao': 'votacao_id'})

df_votos_processed = df_votos_processed.drop_duplicates(subset=['votacao_id', 'deputado_id'])

df_votos_processed['dataHoraVoto'] = pd.to_datetime(df_votos_processed['dataHoraVoto'])

df_votos_processed.head()



,votacao_id,deputado_id,voto,deputado_siglaPartido,deputado_siglaUf,dataHoraVoto
0,106701-223,204379,Sim,MDB,AP,2025-11-05 12:41:17
1,106701-223,220714,Sim,REPUBLICANOS,AM,2025-11-05 12:31:16
2,106701-223,221328,Sim,PL,SP,2025-11-05 12:32:08
3,106701-223,204560,Sim,PSDB,BA,2025-11-05 12:33:17
4,106701-223,204528,Sim,NOVO,SP,2025-11-05 12:32:50


In [6]:
df_orientacao_processed = df_orientacao[df_orientacao['idVotacao'].isin(plen_vote_ids)]

orientacao_features = [
    'idVotacao',
    'siglaBancada',
    'orientacao'
]

df_orientacao_processed = df_orientacao_processed[orientacao_features]

df_orientacao_processed = df_orientacao_processed.rename(columns={'idVotacao': 'votacao_id'})

df_orientacao_processed.head()

,votacao_id,siglaBancada,orientacao
32,2520670-66,PSB,NaN
33,2520670-66,PL,Não
34,2520670-66,Governo,Sim
35,2520670-66,Bl UniPpPsd...,Sim
36,2520670-66,PDT,Sim


In [7]:
assert df_votos_processed.duplicated(subset=['votacao_id', 'deputado_id']).sum() == 0
assert df_votos_processed['votacao_id'].isin(plen_vote_ids).all()
print("✓ All checks passed")

df_deputados_processed.to_parquet(f'{PROCESSED_DIR}deputados.parquet', index=False)
df_votacoes_processed.to_parquet(f'{PROCESSED_DIR}votacoes.parquet', index=False)
df_votos_processed.to_parquet(f'{PROCESSED_DIR}votos.parquet', index=False)
df_orientacao_processed.to_parquet(f'{PROCESSED_DIR}orientacoes.parquet', index=False)

for name in ['deputados', 'votacoes', 'votos', 'orientacoes']:
    df = pd.read_parquet(f'{PROCESSED_DIR}{name}.parquet')
    print(f"{name}: {df.shape}")

✓ All checks passed
deputados: (551, 7)
votacoes: (429, 9)
votos: (171217, 6)
orientacoes: (4126, 3)
